# 📊 Credit Risk Dataset — Data Cleaning & ETL Pipeline
**Project:** Credit Risk Analytics Dashboard  
**Tools:** Python (Pandas, NumPy, SQLAlchemy), PostgreSQL, Power BI  

---

## 1. Import Libraries & Load Raw Dataset
In this section, we import essential Python libraries and read the raw Kaggle Credit Risk dataset.

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(r"C:\Users\ASUS\Desktop\credit\credit_risk_dataset.csv")
df.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


## 2. Exploratory Data Analysis (EDA) & Data Inspection
Checking dataset dimensions, column data types, missing values, and key summary statistics.

In [3]:
df.shape

(32581, 12)

In [4]:
df.duplicated().sum()

np.int64(165)

In [5]:
df = df.drop_duplicates()

In [6]:
df.shape

(32416, 12)

In [7]:
df.dtypes

person_age                      int64
person_income                   int64
person_home_ownership             str
person_emp_length             float64
loan_intent                       str
loan_grade                        str
loan_amnt                       int64
loan_int_rate                 float64
loan_status                     int64
loan_percent_income           float64
cb_person_default_on_file         str
cb_person_cred_hist_length      int64
dtype: object

In [8]:
# 1. Yaş üzrə məntiqsiz dəyərlərin filterlənməsi (məs: yaş < 100)
df = df[df['person_age'] < 100]

# 2. İş stajı üzrə məntiqsiz dəyərlərin filterlənməsi (məs: staj < 60 il)
df = df[df['person_emp_length'] < 60]

# 3. İş stajının yaşdan böyük olması halının qarşısını almaq
df = df[df['person_emp_length'] < df['person_age']]

In [9]:
df.shape

(31522, 12)

In [10]:
df.isnull().sum()

person_age                       0
person_income                    0
person_home_ownership            0
person_emp_length                0
loan_intent                      0
loan_grade                       0
loan_amnt                        0
loan_int_rate                 3027
loan_status                      0
loan_percent_income              0
cb_person_default_on_file        0
cb_person_cred_hist_length       0
dtype: int64

## 3. Data Cleaning & Missing Value Imputation
- Filling missing employment length (`person_emp_length`) using column median.
- Imputing missing interest rates (`loan_int_rate`) based on median interest rate per `loan_grade`.

In [11]:
# Mediana (median) ilə doldurmaq daha təhlükəsizdir (outlier-lərdən təsirlənmir)
df['person_emp_length'] = df['person_emp_length'].fillna(df['person_emp_length'].median())
df['loan_int_rate'] = df['loan_int_rate'].fillna(df['loan_int_rate'].median())

In [12]:
df.shape[0]

31522

In [13]:
df['cb_person_default_on_file'] = df['cb_person_default_on_file'].map({'Y': 1, 'N': 0})

## 4. Feature Engineering & Categorization
Creating aggregated dimension columns to power interactive Power BI slicers:
- **`age_group`**: `18-25`, `26-35`, `36-50`, `50+`
- **`loan_risk_tier`**: *Aşağı Risk*, *Orta Risk*, *Yüksək Risk* based on loan percent of income.

In [14]:
#Yaş qrupları yaratmaq (age_group)
bins = [0, 25, 35, 50, 100]
labels = ['18-25', '26-35', '36-50', '50+']
df['age_group'] = pd.cut(df['person_age'], bins=bins, labels=labels)

In [15]:
# Aylıq gəlir sütunu ekstrasiya etmək
df['monthly_income'] = (df['person_income'] / 12).round(2)

In [16]:
# Borc yükü risk səviyyəsi (loan_risk_tier)
def categorize_loan_risk(pct):
    if pct < 0.20:
        return 'Aşağı Risk'
    elif pct <= 0.40:
        return 'Orta Risk'
    else:
        return 'Yüksək Risk'

df['loan_risk_tier'] = df['loan_percent_income'].apply(categorize_loan_risk)

In [17]:
# Nəticəyə baxış
df[['person_age', 'age_group', 'monthly_income', 'loan_percent_income', 'loan_risk_tier', 'cb_person_default_on_file']].head()

,person_age,age_group,monthly_income,loan_percent_income,loan_risk_tier,cb_person_default_on_file
1,21,18-25,800.00,0.10,Aşağı Risk,0
2,25,18-25,800.00,0.57,Yüksək Risk,0
3,23,18-25,5458.33,0.53,Yüksək Risk,0
4,24,18-25,4533.33,0.55,Yüksək Risk,1
5,21,18-25,825.00,0.25,Orta Risk,0


In [18]:
df.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length,age_group,monthly_income,loan_risk_tier
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,0,2,18-25,800.00,Aşağı Risk
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,0,3,18-25,800.00,Yüksək Risk
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,0,2,18-25,5458.33,Yüksək Risk
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,1,4,18-25,4533.33,Yüksək Risk
5,21,9900,OWN,2.0,VENTURE,A,2500,7.14,1,0.25,0,2,18-25,825.00,Orta Risk


## 5. Export Cleaned Dataset & PostgreSQL Integration
- Exporting the processed dataset to `cleaned_credit_risk_dataset.csv`.
- Uploading cleaned data into local **PostgreSQL** database instance.

In [21]:
df.to_csv('cleaned_credit_risk_dataset.csv', index=False, encoding='utf-8-sig')